# 02 · RAG Query Pipeline

This notebook wires together the full retrieval-augmented generation pipeline:

1. Authenticate with OCI GenAI (Cohere embeddings)
2. Connect to Oracle Autonomous Database 23ai
3. Ingest embedded chunks into the native VECTOR column
4. Run a LangChain RAG chain backed by Cohere Command R
5. Interactive query loop

**Prerequisites**: Run `01_document_ingestion.ipynb` first to produce `all_chunks`.

**Credentials**: Set the environment variables below before running. Never paste secrets directly into notebook cells.

In [ ]:
import os

# Verify all required secrets are present
required_env = [
    "OCI_USER", "OCI_FINGERPRINT", "OCI_TENANCY", "OCI_REGION", "OCI_KEY_FILE",
    "ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN",
    "ORACLE_WALLET_LOCATION", "ORACLE_WALLET_PASSWORD",
]
missing = [k for k in required_env if not os.getenv(k)]
if missing:
    print(f"WARNING: Missing env vars: {missing}")
    print("Set them with: export VAR=value  or via a .env file + python-dotenv")
else:
    print("All required environment variables are set.")

## 1 · OCI GenAI Authentication

We inject the OCI client directly into LangChain's `OCIGenAIEmbeddings` to bypass
file-based config lookup — this is the reliable pattern for notebook environments.

In [ ]:
import oci
from oci.generative_ai_inference import GenerativeAiInferenceClient
from langchain_community.embeddings import OCIGenAIEmbeddings

oci_config = {
    "user":        os.environ["OCI_USER"],
    "fingerprint": os.environ["OCI_FINGERPRINT"],
    "tenancy":     os.environ["OCI_TENANCY"],
    "region":      os.environ["OCI_REGION"],
    "key_file":    os.environ["OCI_KEY_FILE"],
}

CHICAGO_ENDPOINT = "https://inference.generativeai.us-chicago-1.oci.oraclecloud.com"

inference_client = GenerativeAiInferenceClient(
    oci_config,
    service_endpoint=CHICAGO_ENDPOINT,
)

embeddings = OCIGenAIEmbeddings(
    model_id="cohere.embed-english-v3.0",   # 1024 dimensions
    compartment_id=oci_config["tenancy"],
    client=inference_client,
)

# Smoke-test: embed a single string
test_vec = embeddings.embed_query("PropTech real estate technology")
print(f"Embedding dimension: {len(test_vec)}  (expected 1024)")

## 2 · Oracle Autonomous Database 23ai Connection

Oracle DB 23ai stores vectors natively alongside relational data, enabling
hybrid SQL + cosine similarity queries — a key enterprise compliance advantage
over standalone vector databases.

In [ ]:
import oracledb

conn = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASSWORD"],
    dsn=os.environ["ORACLE_DSN"],
    wallet_location=os.environ["ORACLE_WALLET_LOCATION"],
    wallet_password=os.environ["ORACLE_WALLET_PASSWORD"],
)
print(f"Connected to Oracle DB version: {conn.version}")

## 3 · Create Vector Table and Ingest

The `PROPTECH_KNOWLEDGE` table uses Oracle's native `VECTOR(1024, FLOAT32)` column type.
This allows the database to perform approximate nearest-neighbour search using HNSW indexes
without any external vector database.

In [ ]:
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS PROPTECH_KNOWLEDGE PURGE")
cursor.execute("""
    CREATE TABLE PROPTECH_KNOWLEDGE (
        id        VARCHAR2(64)       PRIMARY KEY,
        text      CLOB,
        metadata  JSON,
        embedding VECTOR(1024, FLOAT32)
    )
""")
conn.commit()
cursor.close()
print("Table PROPTECH_KNOWLEDGE created.")

In [ ]:
from langchain_community.vectorstores import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy

vector_store = OracleVS(
    client=conn,
    embedding_function=embeddings,
    table_name="PROPTECH_KNOWLEDGE",
    distance_strategy=DistanceStrategy.COSINE,
    params={"embedding_dim": 1024},
)

# all_chunks comes from notebook 01 — re-run that cell if needed
vector_store.add_documents(all_chunks)
print(f"Ingested {len(all_chunks)} chunks into Oracle 23ai.")

## 4 · Build the RAG Chain

LangChain's `create_retrieval_chain` connects:
- **Retriever**: Oracle vector similarity search (top-k chunks)
- **LLM**: Cohere Command R via OCI GenAI (Chicago region)
- **Prompt**: System instruction with injected context

In [ ]:
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOCIGenAI(
    model_id="cohere.command-r-08-2024",
    service_endpoint=CHICAGO_ENDPOINT,
    compartment_id=oci_config["tenancy"],
    client=inference_client,
    model_kwargs={"max_tokens": 1000, "temperature": 0.3},
)

system_prompt = (
    "You are a knowledgeable PropTech assistant. "
    "Answer the question using only the context provided below. "
    "If the context does not contain enough information, say so rather than guessing.\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

doc_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(vector_store.as_retriever(search_kwargs={"k": 4}), doc_chain)
print("RAG chain ready.")

## 5 · Interactive Query Loop

In [ ]:
print("PropTech AI Assistant — type 'exit' to quit\n")

while True:
    user_input = input("You: ").strip()
    if not user_input or user_input.lower() in {"exit", "quit"}:
        break
    result = qa_chain.invoke({"input": user_input})
    print(f"\nAI: {result['answer']}\n" + "-" * 60)